# 1.8 Audit of the 80 ECDA rows left out of the centre layer

1_05 drops the 80 rows of the Oct-2026 ECDA list whose `centre_code` is "na". If those were real,
operating centres missing from the layer, supply would be understated, and if they were missing in
particular places the subzone comparisons (H-c) would be biased. This notebook follows Jessie's
four checks in order:

1. **Locate** every excluded row (by the retained centre at the same postal code, or OneMap) and
   classify it as a duplicate of a retained centre, possibly missing, or not locatable.
2. **Verify** the 40 possibly missing records one by one against ECDA's upcoming-centres list and
   operator pages (evidence gathered by hand on 2026-11-01): planned, duplicate, operating-but-absent
   (one centre, added back in 1_05) or unverified. The planned centres with a location become layer
   `planned` (13) of the centre GeoPackage.
3. **Test spatial missingness**: are the 40 possibly missing records randomly scattered among all
   observed centre locations, or concentrated (random-labelling permutation test, 9,999 permutations,
   seed 4015, plus an exact test for Tengah)?
4. **Sensitivity**: how much would subzone supply change if the unverified records were real
   centres (plausible and upper-bound scenarios)?

Original scripts: `scripts/locate_excluded_centres_2026.py`, `excluded_centres_verification.py`,
`excluded_centres_missingness.py`, `unverified_centres_sensitivity.py` (all Jessie).

**Run order.** Sections 1 and 4 read tables produced later in the pipeline: `output/subzone_lisa_hc.csv`
(H-c LISA labels, stage 2) and `output/subzone_stats.csv` (1_08), both read-only. Run this notebook
after 1_08 and the ESDA stage, or with their committed outputs in place. Section 1 is served from the
OneMap cache (`ALLOW_ONEMAP = False`): three address queries that returned no result on 2026-11-01
are not in the cache (it keeps hits only) and are treated as "no result" rather than re-queried.

**Inputs**
- `data/CentreAttributes_2026.csv`, `data/MasterPlanSubZones_2019.geojson`.
- `output/childcare_centres_2026.gpkg`, layer `centres` (1_05).
- `output/onemap_geocode_cache.csv` (shared with 1_05).
- `output/subzone_lisa_hc.csv` (stage 2, read only), `output/subzone_stats.csv` (1_08, read only).

**Outputs**
- `output/excluded_centres_2026.csv` — 80 rows with location, subzone, retained centres at the same
  postal code, `qa_status` and the H-c labels.
- `output/excluded_centres_verification_2026.csv` — the 80 rows with `status`, evidence and source.
- `output/childcare_centres_2026.gpkg`, layer `planned` — 13 upcoming ECDA centres with a location
  (read by 1_07 and the pressure-index notebooks).
- `output/excluded_centres_missingness_by_area.csv` — missing vs retained counts by planning area and subzone.
- `output/childcare_centres_2026_sensitivity.gpkg`, layers `plausible` and `upper_bound`;
  `output/unverified_centres_sensitivity_by_subzone.csv`.

In [1]:
import sys
from pathlib import Path

import time

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from scipy.spatial import cKDTree
from scipy.stats import hypergeom

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

# changed 2026-11-10: paths through bt4015.paths.
SRC = DATA / "CentreAttributes_2026.csv"
SUBZONES = DATA / "MasterPlanSubZones_2019.geojson"
LAYER = OUT / "childcare_centres_2026.gpkg"        # layer "centres" read, layer "planned" written
CACHE = OUT / "onemap_geocode_cache.csv"
LISA = OUT / "subzone_lisa_hc.csv"                 # stage 2 output, read only
STATS = OUT / "subzone_stats.csv"                  # 1_08 output, read only
EXCLUDED_CSV = OUT / "excluded_centres_2026.csv"
VERIFICATION_CSV = OUT / "excluded_centres_verification_2026.csv"
MISSINGNESS_CSV = OUT / "excluded_centres_missingness_by_area.csv"
SENSITIVITY_GPKG = OUT / "childcare_centres_2026_sensitivity.gpkg"
SENSITIVITY_CSV = OUT / "unverified_centres_sensitivity_by_subzone.csv"

URL = "https://www.onemap.gov.sg/api/common/elastic/search"
ALLOW_ONEMAP = False        # added 2026-11-10: False = no network; uncached queries count as "no result"
VERIFIED_ON = "2026-11-01"  # date of the manual evidence in section 2
PERMS, CSR_SIMS, SEED, KS = 9999, 999, 4015, (1, 5)   # permutations, CSR simulations, seed, Cuzick-Edwards k

## 1. Locate the 80 excluded rows

Three mutually exclusive groups: `invalid_postal` (postal code 000000; located from the block and
street, or the HDB project name, read off `centre_address` by hand), `postal_in_layer` (a retained
centre already has that postal code; its point is reused and the row is a duplicate), and
`postal_absent` (a valid postal code with no retained centre; geocoded by postal code). A OneMap hit
on a postal code the layer already has is the same building as a retained centre (in every such
case the same operator), so it also counts as a duplicate.

In [2]:
# Search text for the postal-000000 rows, read off centre_address by hand. Block + street
# where one is given, otherwise the HDB project name. None = nothing searchable.
INVALID_QUERIES = {
    "152,Towner Road": "152 TOWNER ROAD",
    "Dakota One": "DAKOTA ONE",
    "Yishun Boardwalk": "YISHUN BOARDWALK",
    "750,Jurong West Street 73": "750 JURONG WEST STREET 73",
    "N1 C25 Marsiling Grove": "MARSILING GROVE",
    "467B,Yishun Ave 6": "467B YISHUN AVENUE 6",
    "WOODLANDS N3 C16 Woodgrove Ascent": "WOODGROVE ASCENT",
    "MK16 Lot No": None,
    "PLC6 - Parc Woods": "PARC WOODS",
    "PAC5": None,
    "470B,SERANGOON CRESCENT": "470B SERANGOON CRESCENT",
    "N1 C6 Eastlink 2": "EASTLINK II",
    "76A,Circuit Road": "76A CIRCUIT ROAD",
    "Blk 665,Tampines St 64": "665 TAMPINES STREET 64",
    "646B,Tampines St 62": "646B TAMPINES STREET 62",
    "489C,Tampines Street 45": "489C TAMPINES STREET 45",
    "691D,Woodlands Drive 73": "691D WOODLANDS DRIVE 73",
    "223A,Sumang Lane": "223A SUMANG LANE",
    "445A,Northshore Drive": "445A NORTHSHORE DRIVE",
    "C59,Kent Heights": "KENT HEIGHTS",
    "Bishan Towers": "BISHAN TOWERS",
    "326,Ubi Avenue 1": "326 UBI AVENUE 1",
    "Bidadari C13": "BARTLEY GREENRISE",
    "443A,Bukit Batok West Avenue 8": "443A BUKIT BATOK WEST AVENUE 8",
}


def onemap(query, cache, want_postal=None):
    """Cached OneMap search -> (lat, lon, postal, address); a tuple of Nones when nothing is found."""
    if query in cache:
        return cache[query]
    # changed 2026-11-10: the cache keeps hits only, so the three queries that returned nothing on
    # 2026-11-01 (751897, 640991, 445A NORTHSHORE DRIVE) are not in it. Offline reruns treat an
    # uncached query as "no result" instead of calling the API; set ALLOW_ONEMAP = True to re-query.
    if not ALLOW_ONEMAP:
        print(f"  not in cache, treated as no result: {query!r}")
        cache[query] = (None, None, None, None)
        return cache[query]
    for attempt in range(6):
        r = requests.get(URL, params={"searchVal": query, "returnGeom": "Y",
                                      "getAddrDetails": "Y", "pageNum": 1},
                         headers={"User-Agent": "curl/8.4.0"}, timeout=(5, 10))
        if r.status_code == 429:
            time.sleep(4 + 3 * attempt)
            continue
        res = r.json().get("results", [])
        if want_postal:
            res = [x for x in res if x.get("POSTAL") == want_postal]
        hit = (float(res[0]["LATITUDE"]), float(res[0]["LONGITUDE"]), res[0]["POSTAL"],
               res[0]["ADDRESS"]) if res else (None, None, None, None)
        cache[query] = hit
        time.sleep(2.2)
        return hit
    raise RuntimeError(f"OneMap kept rate-limiting on {query!r}")


# 1. The 80 "na" rows and the retained ECDA-listed centres (the verified addition is itself one of the 80).
raw = pd.read_csv(SRC, dtype=str)
excluded = raw[raw["centre_code"].str.lower() == "na"].copy()
retained = gpd.read_file(LAYER, layer="centres")
retained = retained[retained["record_source"] == "ecda_listed"]
retained_by_postal = retained.groupby("postal_code")
excluded["group"] = [
    "invalid_postal" if p == "000000"
    else "postal_in_layer" if p in retained_by_postal.groups else "postal_absent"
    for p in excluded["postal_code"]]
print(f"excluded rows: {len(excluded)}; retained centres: {len(retained)}")
print(excluded["group"].value_counts().to_string())

# 2. The shared OneMap cache, as (lat, lon, postal, address) tuples.
cache_df = pd.read_csv(CACHE, dtype=str)
cache = {q: (float(la), float(lo), p, a)
         for q, la, lo, p, a in cache_df[["query", "lat", "lon", "postal", "address"]].values}
print(f"cached OneMap queries: {len(cache)}")

excluded rows: 80; retained centres: 1795
group
postal_in_layer    31
postal_absent      25
invalid_postal     24
cached OneMap queries: 346


In [3]:
# 1. One location per excluded row: reuse the retained centre's point, or look the address up.
rows = []
for _, r in excluded.iterrows():
    out = {"query": None, "lat": None, "lon": None, "onemap_address": None,
           "retained_centres_same_postal": None}
    if r["group"] == "postal_in_layer":
        same = retained_by_postal.get_group(r["postal_code"])
        pt = same.to_crs(4326).geometry.iloc[0]
        out.update(lat=pt.y, lon=pt.x, location_source="retained_centre_point",
                   retained_centres_same_postal="; ".join(
                       same["centre_code"] + " " + same["centre_name"]))
    else:
        if r["group"] == "postal_absent":
            q, want = r["postal_code"], r["postal_code"]
        else:
            key = next((k for k in INVALID_QUERIES if r["centre_address"].startswith(k)), None)
            q, want = INVALID_QUERIES.get(key), None
        if q:
            la, lo, p, a = onemap(q, cache, want)
            out.update(query=q, lat=la, lon=lo, onemap_address=a)
        out["location_source"] = "onemap" if out["lat"] is not None else "not_located"
    rows.append({**r[["tp_code", "centre_name", "service_model", "organisation_description",
                      "centre_address", "postal_code", "last_updated", "group"]], **out})

# 2. Save the cache (hits only) so 1_05 and later reruns share it.
pd.DataFrame([(q, *v) for q, v in cache.items() if v[0] is not None],
             columns=["query", "lat", "lon", "postal", "address"]).to_csv(CACHE, index=False)

# 3. QA status: duplicate of a retained centre, possibly missing, or not locatable.
located = pd.DataFrame(rows)
located["onemap_postal"] = located["onemap_address"].str.extract(r"(\d{6})$")[0]
located["qa_status"] = "possible_missing"
located.loc[located["onemap_postal"].isin(retained_by_postal.groups), "qa_status"] = "duplicate"
located.loc[located["group"] == "postal_in_layer", "qa_status"] = "duplicate"
located.loc[located["location_source"] == "not_located", "qa_status"] = "not_located"

# 4. Subzone and the H-c LISA labels of that subzone, then write.
excluded_pts = gpd.GeoDataFrame(located, geometry=gpd.points_from_xy(located["lon"], located["lat"]), crs=4326)
sz_wgs = gpd.read_file(SUBZONES).to_crs(4326)[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "geometry"]]
excluded_pts = gpd.sjoin(excluded_pts, sz_wgs, how="left", predicate="within").drop(columns="index_right")
lisa = pd.read_csv(LISA)[["SUBZONE_C", "children_0_9", "centres_n",
                          "centres_per_1000_children", "bv_walk_q", "bv_sup_q", "bv_sup_p"]]
excluded_pts = excluded_pts.merge(lisa, on="SUBZONE_C", how="left")
excluded_pts.drop(columns="geometry").to_csv(EXCLUDED_CSV, index=False)
print(excluded_pts["location_source"].value_counts().to_string())
print(excluded_pts["qa_status"].value_counts().to_string())
print(f"wrote {EXCLUDED_CSV.relative_to(ROOT)} ({len(excluded_pts)} rows)")

  not in cache, treated as no result: '751897'
  not in cache, treated as no result: '640991'
  not in cache, treated as no result: '445A NORTHSHORE DRIVE'
location_source
onemap                   44
retained_centre_point    31
not_located               5
qa_status
possible_missing    40
duplicate           35
not_located          5
wrote output/excluded_centres_2026.csv (80 rows)


## 2. Verification of the 80 rows (evidence gathered 2026-11-01)

The 40 possibly missing records were each checked against ECDA's upcoming-preschools page, the
operators' own centre pages, HDB precinct brochures and the retained centres' addresses. Statuses:

- `planned` — on ECDA's upcoming-centres list, so not operating at 30 Sep 2026;
- `duplicate` — same block and unit, same named HDB project, or a stated extension of a centre
  already in the layer;
- `operational_absent` — operating per the operator's page but not in the ECDA list under any
  code (one centre, PCF Sparkletots @ Yuhua Blk 285C, TP1356; 1_05 adds it back);
- `unverified` — no source found; `likely` gives the best guess and is not used as evidence.

`EVIDENCE` is keyed by the OneMap postal code of each record (unique among the 40); `OVERRIDES` by
`tp_code` covers duplicate or unlocated records that ECDA nevertheless lists as upcoming.

In [4]:
ECDA = "https://ecda.gov.sg/parents/preschool-search/upcoming-preschools"
PCF_UP = "https://pcfsparkletots.org.sg/upcoming-preschools/"
LAYER_SRC = "data/CentreAttributes_2026.csv (retained centre address)"

# postal: (status, likely, evidence, source)
EVIDENCE = {
    "760936": ("planned", "", "ECDA upcoming: My First Skool, Blk 936 Yishun Central, est. Q4 2026", ECDA),
    "829910": ("planned", "", "ECDA upcoming: E-Bridge Pre-school, 82 Punggol Way #03-27 to #03-37, est. Q4 2026", ECDA),
    "690135": ("planned", "", "ECDA upcoming: PCF Sparkletots @ Tengah Blk 135 (CC), est. Q4 2026; PCF upcoming list same", ECDA),
    "531214": ("planned", "", "ECDA upcoming: MY World Preschool, Blk 214A Tampines Road #02-132, est. Q4 2026", ECDA),
    "532182": ("planned", "", "ECDA upcoming: PCF Sparkletots @ Geylang Serai Blk 182B (CC), est. Q4 2026 (PCF says Q1 2027)", ECDA),
    "521899": ("planned", "", "ECDA upcoming: extension to PCF Sparkletots @ Tampines West Blk 890A (CC), Blk 899A, est. Q4 2026", ECDA),
    "320152": ("planned", "", "ECDA upcoming: My First Skool, Blk 152 Towner Road #01-01, est. Q1 2027", ECDA),
    "694216": ("planned", "", "ECDA upcoming: My First Skool, Blk 216D / 217A Tengah Garden Avenue, est. Q1 2027", ECDA),
    "521634": ("planned", "", "ECDA upcoming: PCF Sparkletots @ Tampines Boulevard Blk 634A (CC), est. Q1 2027", ECDA),
    "524859": ("planned", "", "ECDA upcoming: extension to E-Bridge Pre-school at Tampines GreenSpring, Blk 859D, est. Q1 2027", ECDA),
    "211055": ("planned", "", "ECDA upcoming: Kent Height, Blk 55A Owen Road, est. Q1 2027", ECDA),
    "730909": ("duplicate", "", "Same block and unit (909 North Woodlands Way #01-01, UrbanVille) as PCF Sparkletots @ Marsiling Blk 909 (CC)", LAYER_SRC),
    "732894": ("duplicate", "", "Same block and unit (894D Woodlands Dr 50 #01-19) as PCF Sparkletots @ Woodgrove Blk 894D (CC)", LAYER_SRC),
    "791415": ("duplicate", "", "Same block (416A Fernvale Link) and service (DS) as PCF Sparkletots @ Buangkok-Fernvale South Blk 416A (DS)", LAYER_SRC),
    "691302": ("duplicate", "", "Same HDB project (Parc Residences @ Tengah) as ST0417 PCF Sparkletots @ Brickland Blk 300 (CC)", LAYER_SRC),
    "690307": ("duplicate", "", "Same HDB project (Parc Clover @ Tengah) as ST0405 PCF Sparkletots @ Brickland Blk 308B (CC)", LAYER_SRC),
    "382020": ("duplicate", "", "Same HDB project (Kallang Breeze) as PCF Sparkletots @ Kreta Ayer-Kim Seng Blk 21 (CC)", LAYER_SRC),
    "732189": ("duplicate", "", "Same HDB project (Marsiling Grove) as My First Skool, 187 Marsiling Road #03-11", LAYER_SRC),
    "762467": ("duplicate", "", "Same HDB project (Casa Spring @ Yishun) as NT0605 My First Skool, 464 Yishun Ave 6", LAYER_SRC),
    "522489": ("duplicate", "", "Record states it is an extension of the My First Skool main centre at Blk 487C Tampines St 45, which is in the layer", "data/CentreAttributes_2026.csv (record's own address)"),
    "603285": ("operational_absent", "",
               "Operator page: 'PCF Sparkletots Preschool @ Yuhua Blk 285C (KN)', Blk 285C Toh Guan Rd #01-92, Mon-Fri 8am-5pm, no closure notice; "
               "same unit as this record; also in the 2021 layer; not in the 2026 ECDA list under any centre code",
               "https://pcfsparkletots.org.sg/our-preschools/pcf-sparkletots-bukit-batok-east-blk-285c"),
    "821223": ("unverified", "duplicate", "Matilda Edge (Blks 220-224 Sumang Lane) is served by PCF Sparkletots @ Punggol West Blk 222 (CC), Blk 222 Sumang Lane MSCP, 160 m away; not on ECDA upcoming list", "https://pcf.org.sg/sparkletots/our-preschools/pcf-sparkletots-punggol-west-blk-222"),
    "571156": ("unverified", "duplicate", "Bishan Towers (156A); My First Skool at 156C Bishan St 11, 16 m away", LAYER_SRC),
    "752101": ("unverified", "duplicate", "EastCreek @ Canberra (101B); Skool4Kidz at 101 Canberra St, 46 m away", LAYER_SRC),
    "532470": ("unverified", "duplicate", "Hougang Parkview (470B); PCF Sparkletots Blk 470C Upper Serangoon Cres, 55 m away", LAYER_SRC),
    "651443": ("unverified", "duplicate", "West Ridges (443A); E-Bridge at 443 Bukit Batok West Ave 8 MSCP, 68 m away", LAYER_SRC),
    "750333": ("unverified", "duplicate", "My First Skool at 331 Sembawang Close, 66 m away", LAYER_SRC),
    "731372": ("unverified", "duplicate", "Woodgrove Ascent (372A); My First Skool at 375 Woodlands Ave 1, 163 m away", LAYER_SRC),
    "751117": ("unverified", "duplicate", "Eastlink II (117A); My First Skool at 120 Canberra Cres MSCP, 126 m away", LAYER_SRC),
    "531996": ("unverified", "duplicate", "Buangkok Tropica (996A); My First Skool at 997 Buangkok Cres MSCP, 147 m away", LAYER_SRC),
    "659841": ("unverified", "historical", "My First Skool was at this postal code in 2021; My First Skool's site lists no centre there now (nearest Blk 260, in the layer)", "https://www.myfirstskool.com/our-centres/blk-260-bukit-batok/"),
    "390054": ("unverified", "", "No preschool found at 54 Cassia Crescent; My First Skool NT0571 at 56 Cassia Crescent is in the layer", LAYER_SRC),
    "392088": ("unverified", "", "No preschool found at Dakota One", ""),
    "761338": ("unverified", "", "HDB precinct brochure shows a preschool in Yishun Boardwalk; no operator or opening found", "https://www.mynicehome.gov.sg/wp-content/uploads/2025/03/RS-LivinginYishun-YishunBoardwalk.pdf"),
    "361216": ("unverified", "", "HDB precinct brochure shows a preschool in Bartley GreenRise; not on ECDA upcoming list (30 Sep 2026)", "https://www.mynicehome.gov.sg/wp-content/uploads/RS-LivinginBidadari-BartleyGreenRise.pdf"),
    "734691": ("unverified", "", "No preschool found in Admiralty Flora", ""),
    "684485": ("unverified", "", "No preschool found at 485D Choa Chu Kang Ave 5", ""),
    "520665": ("unverified", "", "No preschool found in Tampines GreenCrest", ""),
    "522646": ("unverified", "", "No preschool found in Tampines GreenFoliage", ""),
    "730586": ("unverified", "", "No preschool found at 586 Woodlands Dr 16; PCF Sparkletots @ Admiralty Blk 585 is 117 m away", LAYER_SRC),
}

# Duplicate or unlocated records that ECDA lists as upcoming (checked against the same page).
OVERRIDES = {
    "TP1371": ("planned", "", "ECDA upcoming: PCF Sparkletots Preschool, Blk 483 Tampines St 43 #01-202, est. Q4 2027 (a new service in the building of the retained PCF KN centre)", ECDA),
    "TP1372": ("planned", "", "ECDA upcoming: PCF Sparkletots Preschool, Blk 868A Tampines Ave 8 #01-534, est. Q4 2027 (a new service in the building of the retained PCF KN centre)", ECDA),
    "TP1198": ("planned", "", "ECDA upcoming: PCF Sparkletots @ Sembawang Central (CC), 3A Canberra View, est. Q1 2027; PCF upcoming list same", ECDA),
}
IN_LAYER = {"TP1356"}  # must match VERIFIED_ADDITIONS in 1_05
print(f"evidence for {len(EVIDENCE)} possibly missing records, {len(OVERRIDES)} overrides, {len(IN_LAYER)} added to the layer")

evidence for 40 possibly missing records, 3 overrides, 1 added to the layer


In [5]:
# 1. Re-read section 1's table and check the evidence covers exactly the possibly-missing records.
review = pd.read_csv(EXCLUDED_CSV, dtype=str)
review["onemap_postal"] = review["onemap_address"].str.extract(r"(\d{6})$")[0]
pm = review["qa_status"] == "possible_missing"
assert review.loc[pm, "onemap_postal"].is_unique and set(review.loc[pm, "onemap_postal"]) == set(EVIDENCE)


def row_status(r):
    """(status, likely, evidence, source) for one excluded record."""
    if r["tp_code"] in OVERRIDES:
        return OVERRIDES[r["tp_code"]]
    if r["qa_status"] == "possible_missing":
        return EVIDENCE[r["onemap_postal"]]
    if r["qa_status"] == "duplicate":
        same = r["retained_centres_same_postal"]
        if pd.isna(same):
            same = f"the retained centre at postal code {r['onemap_postal']}"
        return ("duplicate", "", f"Same building and operator as {same}", LAYER_SRC)
    return ("unverified", "", "No OneMap result for the address and not on the ECDA upcoming list", "")


# 2. Status per row, then the verification table.
ev = pd.DataFrame(review.apply(row_status, axis=1).tolist(), index=review.index,
                  columns=["status", "likely", "evidence", "source"])
verified = pd.concat([review, ev], axis=1).assign(verified_on=VERIFIED_ON)
verified["in_layer"] = verified["tp_code"].isin(IN_LAYER)
verified = verified[["tp_code", "centre_name", "service_model", "centre_address", "onemap_address",
                     "lat", "lon", "SUBZONE_N", "PLN_AREA_N", "last_updated", "qa_status", "status",
                     "likely", "in_layer", "evidence", "source", "verified_on"]]
verified.sort_values(["status", "PLN_AREA_N"]).to_csv(VERIFICATION_CSV, index=False)
print(pd.crosstab(verified["qa_status"], verified["status"], margins=True).to_string())
print(f"in_layer: {verified['in_layer'].sum()}")
print(f"wrote {VERIFICATION_CSV.relative_to(ROOT)} ({len(verified)} rows)")

status            duplicate  operational_absent  planned  unverified  All
qa_status                                                                
duplicate                33                   0        2           0   35
not_located               0                   0        1           4    5
possible_missing          9                   1       11          19   40
All                      42                   1       14          23   80
in_layer: 1
wrote output/excluded_centres_verification_2026.csv (80 rows)


### 2.1 The planned-centre layer

Planned centres with a location become layer `planned` of `childcare_centres_2026.gpkg`, with
the expected opening quarter parsed from the evidence. 1_05 only rewrites layer `centres`, so this
layer survives a rebuild of the main layer. Used by 1_07 (walk times to planned centres) and the
pressure-index "future" scenario.

In [6]:
# 1. Planned records with coordinates, as points in SVY21.
planned = verified[(verified["status"] == "planned") & verified["lat"].notna()].copy()
planned["expected_opening"] = planned["evidence"].str.extract(r"est\. (Q\d \d{4})")[0]
planned["record_source"] = "planned"
planned = gpd.GeoDataFrame(planned, geometry=gpd.points_from_xy(planned["lon"].astype(float), planned["lat"].astype(float)),
                           crs=4326).to_crs(3414)

# 2. Subzone fields from the polygons (the CSV's names came from section 1; re-derive with the code and region).
sz = gpd.read_file(SUBZONES).to_crs(3414)[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "REGION_N", "geometry"]]
planned = gpd.sjoin(planned.drop(columns=["SUBZONE_N", "PLN_AREA_N"]), sz, predicate="within")
planned = planned[["tp_code", "record_source", "centre_name", "service_model", "centre_address",
                   "onemap_address", "lon", "lat", "SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "REGION_N",
                   "expected_opening", "evidence", "source", "verified_on", "geometry"]]
planned.to_file(LAYER, layer="planned", driver="GPKG")
print(f"planned layer: {len(planned)} centres "
      f"({(verified['status'] == 'planned').sum() - len(planned)} planned without a location left out)")
print(planned[["tp_code", "PLN_AREA_N", "SUBZONE_N", "expected_opening"]].to_string(index=False))

planned layer: 13 centres (1 planned without a location left out)
tp_code PLN_AREA_N      SUBZONE_N expected_opening
 TP1345   TAMPINES TAMPINES NORTH          Q1 2027
 TP0777    PUNGGOL     NORTHSHORE          Q4 2026
 TP0639    HOUGANG       TAI SENG          Q4 2026
 TP1051     TENGAH         GARDEN          Q1 2027
 TP0657    KALLANG      BENDEMEER          Q1 2027
 TP0641    HOUGANG  LORONG AH SOO          Q4 2026
 TP0977     YISHUN YISHUN CENTRAL          Q4 2026
 TP1433     TENGAH     PLANTATION          Q4 2026
 TP1372   TAMPINES  TAMPINES WEST          Q4 2027
 TP1371   TAMPINES  TAMPINES EAST          Q4 2027
 TP1353   TAMPINES  TAMPINES WEST          Q1 2027
 TP1348   TAMPINES  TAMPINES WEST          Q4 2026
 TP0658    KALLANG   KAMPONG JAVA          Q1 2027


## 3. Spatial missingness of the 40 possibly missing records

Null hypothesis (random labelling): the 40 missing records are a random draw from all 1,835
observed centre locations (1,795 retained + 40). Statistics, each compared with its distribution
over 9,999 label permutations (seed 4015), one-sided towards clustering:

- `G_pa`, `G_sz` — likelihood-ratio G = 2 Σ o ln(o/e) of missing counts by planning area / subzone
  (expected = the area's share of all centres × 40): concentration by area.
- `T_1`, `T_5` — Cuzick-Edwards: pairs (missing i, one of i's k nearest centres) where the neighbour
  is also missing: point-level clustering.
- `mean_nn` — mean nearest-neighbour distance among the 40 (small = clustered).

Tengah, the new town where most records are placeholders for centres not yet open, gets an exact
hypergeometric test and the G statistic is repeated without it. Clark-Evans R against complete
spatial randomness inside the land boundary is reported for reference only: centres are not
uniform over Singapore, so CSR overstates clustering, and the retained centres' R shows by how much.

In [7]:
def g_stat(counts, expected):
    """Likelihood-ratio G over the cells with at least one observation."""
    o = counts[counts > 0]
    return 2 * np.sum(o * np.log(o / expected[counts > 0]))


def mean_nn(xy):
    """Mean distance from each point to its nearest other point."""
    d, _ = cKDTree(xy).query(xy, k=2)
    return d[:, 1].mean()


def clark_evans(xy, area):
    """Clark-Evans R: observed mean nearest-neighbour distance over the CSR expectation."""
    return mean_nn(xy) / (0.5 / np.sqrt(len(xy) / area))


# 1. All observed centre locations with a missing / retained label, tagged with their subzone.
rng = np.random.default_rng(SEED)
kept_pts = gpd.read_file(LAYER, layer="centres")
kept_pts = kept_pts.loc[kept_pts["record_source"] == "ecda_listed", ["geometry"]]
missing_rows = pd.read_csv(EXCLUDED_CSV, dtype=str)
missing_rows = missing_rows[missing_rows["qa_status"] == "possible_missing"]
miss_pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(missing_rows["lon"].astype(float),
                                                        missing_rows["lat"].astype(float)), crs=4326).to_crs(3414)
points = pd.concat([kept_pts.assign(missing=0), miss_pts.assign(missing=1)], ignore_index=True)
points = gpd.sjoin(gpd.GeoDataFrame(points, crs=3414), sz[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "geometry"]],
                   how="left", predicate="within")
assert points["SUBZONE_C"].notna().all() and len(points) == len(kept_pts) + len(miss_pts)
lab = points["missing"].to_numpy()
n, m = len(lab), lab.sum()
xy = np.c_[points.geometry.x, points.geometry.y]
print(f"{n} centre locations, {m} missing ({100 * m / n:.2f}%), {len(kept_pts)} retained")

# 2. Missing rates by planning area and subzone (written for the record).
area_rows = []
for level, col in [("planning_area", "PLN_AREA_N"), ("subzone", "SUBZONE_N")]:
    t = points.groupby(col)["missing"].agg(total="size", missing="sum").reset_index()
    t["retained"] = t["total"] - t["missing"]
    t["missing_rate_pct"] = 100 * t["missing"] / t["total"]
    t["expected_missing"] = t["total"] * m / n
    area_rows.append(t.rename(columns={col: "area"}).assign(level=level))
rates = pd.concat(area_rows)[["level", "area", "retained", "missing", "total",
                             "missing_rate_pct", "expected_missing"]]
rates.sort_values(["level", "missing_rate_pct"], ascending=[True, False]).to_csv(MISSINGNESS_CSV, index=False)
print("Planning areas with the most missing records:")
print(rates[rates["level"] == "planning_area"].sort_values("missing", ascending=False).head(6).round(2).to_string(index=False))

1835 centre locations, 40 missing (2.18%), 1795 retained
Planning areas with the most missing records:
        level      area  retained  missing  total  missing_rate_pct  expected_missing
planning_area WOODLANDS        96        6    102              5.88              2.22
planning_area  TAMPINES       126        6    132              4.55              2.88
planning_area   HOUGANG        82        4     86              4.65              1.87
planning_area    TENGAH        14        4     18             22.22              0.39
planning_area SEMBAWANG        59        3     62              4.84              1.35
planning_area   KALLANG        47        3     50              6.00              1.09


In [8]:
# 1. Precomputed structures so each permutation is a few array lookups.
codes = {c: pd.factorize(points[c])[0] for c in ["PLN_AREA_N", "SUBZONE_N"]}
exp = {c: np.bincount(codes[c]) * m / n for c in codes}
_, knn = cKDTree(xy).query(xy, k=max(KS) + 1)
knn = knn[:, 1:]


def stats(l):
    """All test statistics for one labelling l (1 = missing)."""
    idx = np.flatnonzero(l)
    out = {f"G_{'pa' if c == 'PLN_AREA_N' else 'sz'}":
           g_stat(np.bincount(codes[c][idx], minlength=len(exp[c])), exp[c]) for c in codes}
    for k in KS:
        out[f"T_{k}"] = l[knn[idx, :k]].sum()
    out["mean_nn"] = mean_nn(xy[idx])
    return out


# 2. Observed statistics against PERMS random labellings.
obs = stats(lab)
sims = pd.DataFrame([stats(rng.permutation(lab)) for _ in range(PERMS)])
print(f"Random labelling, {PERMS} permutations (pseudo p, one-sided towards clustering)")
for s, v in obs.items():
    more = (sims[s] <= v) if s == "mean_nn" else (sims[s] >= v)
    p = (more.sum() + 1) / (PERMS + 1)
    print(f"  {s:8s} observed {v:10.2f}   null mean {sims[s].mean():10.2f}   "
          f"null 95th/5th {sims[s].quantile(0.05 if s == 'mean_nn' else 0.95):10.2f}   p = {p:.4f}")

Random labelling, 9999 permutations (pseudo p, one-sided towards clustering)
  G_pa     observed      59.72   null mean      36.22   null 95th/5th      49.38   p = 0.0023
  G_sz     observed     122.04   null mean     127.87   null 95th/5th     143.31   p = 0.7339
  T_1      observed       1.00   null mean       2.23   null 95th/5th       5.00   p = 0.8656
  T_5      observed      10.00   null mean       5.93   null 95th/5th      11.00   p = 0.1210
  mean_nn  observed    1017.96   null mean    1459.62   null 95th/5th    1215.70   p = 0.0013


In [9]:
# 1. Tengah: exact hypergeometric test on the planning-area count, Bonferroni over all planning areas.
t = rates[(rates["level"] == "planning_area") & (rates["area"] == "TENGAH")].iloc[0]
p_t = hypergeom.sf(t["missing"] - 1, n, m, t["total"])
n_pa = rates["level"].eq("planning_area").sum()
print(f"Tengah: {t['missing']} of {t['total']} missing ({t['missing_rate_pct']:.1f}%) vs "
      f"{100 * m / n:.2f}% overall; expected {t['expected_missing']:.2f}; "
      f"rate ratio {t['missing_rate_pct'] / (100 * m / n):.1f}; hypergeometric P(X>={t['missing']}) = {p_t:.4f}; "
      f"Bonferroni over {n_pa} planning areas = {min(1, p_t * n_pa):.3f}")

# 2. Does the planning-area concentration survive without Tengah?
keep = (points["PLN_AREA_N"] != "TENGAH").to_numpy()
c, l = pd.factorize(points.loc[keep, "PLN_AREA_N"])[0], lab[keep]
e = np.bincount(c) * l.sum() / len(l)
g0 = g_stat(np.bincount(c[l == 1], minlength=len(e)), e)
gs = np.array([g_stat(np.bincount(c[pl == 1], minlength=len(e)), e)
               for pl in (rng.permutation(l) for _ in range(PERMS))])
print(f"G_pa without Tengah: observed {g0:.2f}, null mean {gs.mean():.2f}, "
      f"p = {((gs >= g0).sum() + 1) / (PERMS + 1):.4f}")

Tengah: 4 of 18 missing (22.2%) vs 2.18% overall; expected 0.39; rate ratio 10.2; hypergeometric P(X>=4) = 0.0005; Bonferroni over 42 planning areas = 0.020


G_pa without Tengah: observed 48.02, null mean 34.76, p = 0.0406


In [10]:
# 1. Clark-Evans against CSR inside the land boundary (reference only; centres are not uniform).
land = sz.geometry.union_all()
minx, miny, maxx, maxy = land.bounds
area = land.area


def csr(k):
    """k uniform random points inside the land polygon (rejection sampling)."""
    out = np.empty((0, 2))
    while len(out) < k:
        c = rng.uniform([minx, miny], [maxx, maxy], size=(4 * k, 2))
        out = np.vstack([out, c[gpd.points_from_xy(c[:, 0], c[:, 1]).within(land)]])
    return out[:k]


r_obs = clark_evans(xy[lab == 1], area)
r_sim = np.array([clark_evans(csr(m), area) for _ in range(CSR_SIMS)])
p_csr = ((r_sim <= r_obs).sum() + 1) / (CSR_SIMS + 1)
print(f"Clark-Evans vs CSR in subzone boundary ({area / 1e6:.0f} km2): R(missing) = {r_obs:.3f}, "
      f"CSR mean {r_sim.mean():.3f}, p = {p_csr:.4f}; "
      f"R(retained) = {clark_evans(xy[lab == 0], area):.3f}")
print(f"wrote {MISSINGNESS_CSV.relative_to(ROOT)} ({len(rates)} rows)")

Clark-Evans vs CSR in subzone boundary (785 km2): R(missing) = 0.460, CSR mean 1.182, p = 0.0010; R(retained) = 0.552
wrote output/excluded_centres_missingness_by_area.csv (280 rows)


## 4. Sensitivity of subzone supply to the unverified records

Of the 23 unverified records, 19 have a location; the other 4 cannot be placed and are left out of
every scenario. Scenarios (centres added to the canonical layer):

- `base` — the layer as built (1,796 centres);
- `plausible` — plus the unverified records not judged likely duplicates (`likely != "duplicate"`);
- `upper_bound` — plus all 19 located unverified records.

Children per subzone come from `output/subzone_stats.csv` (1_08) and the H-c labels from
`output/subzone_lisa_hc.csv` (stage 2), both read only; nothing in the analysis is rerun here. The
question is whether any H-c subzone (supply HL or walk HH) would change category if the unverified
records were real.

In [11]:
# 1. Located unverified records as points with subzone fields, in the layer's column set.
base = gpd.read_file(LAYER, layer="centres")
verif = pd.read_csv(VERIFICATION_CSV, dtype=str)
unver = verif[(verif["status"] == "unverified") & verif["lat"].notna()].copy()
print(f"unverified: {(verif['status'] == 'unverified').sum()}, located: {len(unver)}")

added = gpd.GeoDataFrame(unver, geometry=gpd.points_from_xy(unver["lon"].astype(float), unver["lat"].astype(float)),
                         crs=4326).to_crs(3414)
added = gpd.sjoin(added.drop(columns=["SUBZONE_N", "PLN_AREA_N"]), sz, predicate="within").drop(columns="index_right")
added["lon"], added["lat"] = added["lon"].astype(float), added["lat"].astype(float)
added["record_source"] = "unverified_sensitivity"
added["location_source"] = "onemap_qa"
keep_cols = [c for c in base.columns if c in added.columns]

# 2. Two scenario layers, and centre counts per subzone for each.
scenarios = {"plausible": added[added["likely"].fillna("") != "duplicate"], "upper_bound": added}
counts = {"base": base.groupby("SUBZONE_C").size()}
for name, extra in scenarios.items():
    layer = pd.concat([base, extra[keep_cols]], ignore_index=True)
    gpd.GeoDataFrame(layer, crs=base.crs).to_file(SENSITIVITY_GPKG, layer=name, driver="GPKG")
    counts[name] = layer.groupby("SUBZONE_C").size()
    print(f"{name}: +{len(extra)} -> {len(layer)} centres")

unverified: 23, located: 19
plausible: +10 -> 1806 centres
upper_bound: +19 -> 1815 centres


In [12]:
# 1. Centres per 1,000 children 0-9 under each scenario, with the H-c labels.
stats_sz = pd.read_csv(STATS)[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "children_0_9"]]
lisa_hc = pd.read_csv(LISA)[["SUBZONE_C", "bv_sup_q", "bv_sup_p", "bv_walk_q", "bv_walk_p"]]
sens = stats_sz.merge(pd.DataFrame(counts).fillna(0).astype(int), left_on="SUBZONE_C",
                      right_index=True, how="left").fillna({k: 0 for k in counts})
for k in counts:
    sens[f"per_1000_{k}"] = 1000 * sens[k] / sens["children_0_9"].where(sens["children_0_9"] > 0)
sens["added_upper_bound"] = sens["upper_bound"] - sens["base"]
sens["pct_change_upper_bound"] = 100 * sens["added_upper_bound"] / sens["base"].where(sens["base"] > 0)
sens = sens.merge(lisa_hc, on="SUBZONE_C", how="left")
sens = sens.sort_values(["added_upper_bound", "pct_change_upper_bound"], ascending=False)
sens.to_csv(SENSITIVITY_CSV, index=False)

# 2. Which subzones gain a centre, and what happens to the H-c subzones.
hit = sens[sens["added_upper_bound"] > 0]
print(f"subzones gaining a centre: {len(hit)}")
print(hit[["SUBZONE_N", "PLN_AREA_N", "children_0_9", "base", "plausible", "upper_bound",
           "per_1000_base", "per_1000_upper_bound", "pct_change_upper_bound",
           "bv_sup_q", "bv_walk_q"]].round(2).to_string(index=False))
sig = sens[(sens["bv_sup_q"] == "HL") | (sens["bv_walk_q"] == "HH")]
print("\nH-c subzones (supply HL or walk HH):")
print(sig[["SUBZONE_N", "base", "upper_bound", "bv_sup_q", "bv_sup_p", "bv_walk_q"]].to_string(index=False))
print(f"wrote {SENSITIVITY_GPKG.relative_to(ROOT)} and {SENSITIVITY_CSV.relative_to(ROOT)}")

subzones gaining a centre: 16
        SUBZONE_N    PLN_AREA_N  children_0_9  base  plausible  upper_bound  per_1000_base  per_1000_upper_bound  pct_change_upper_bound bv_sup_q bv_walk_q
   SEMBAWANG EAST     SEMBAWANG        3270.0   9.0        9.0         11.0           2.75                  3.36                   22.22       ns        ns
   TAMPINES NORTH      TAMPINES        8540.0  10.0       12.0         12.0           1.17                  1.41                   20.00       ns        ns
         ALJUNIED       GEYLANG        2140.0  19.0       21.0         21.0           8.88                  9.81                   10.53       ns        ns
 BUKIT BATOK EAST   BUKIT BATOK         580.0   4.0        5.0          5.0           6.90                  8.62                   25.00       ns        ns
         BIDADARI     TOA PAYOH        2370.0   6.0        7.0          7.0           2.53                  2.95                   16.67       ns        HL
     HOUGANG EAST       HOUGANG   

## 5. Files written

In [13]:
import pyogrio
for path in [EXCLUDED_CSV, VERIFICATION_CSV, MISSINGNESS_CSV, SENSITIVITY_CSV]:
    print(f"{path.relative_to(ROOT)}: {len(pd.read_csv(path)):,} rows")
print(f"{LAYER.relative_to(ROOT)} layer planned: {pyogrio.read_info(LAYER, layer='planned')['features']} features")
for layer in ["plausible", "upper_bound"]:
    print(f"{SENSITIVITY_GPKG.relative_to(ROOT)} layer {layer}: {pyogrio.read_info(SENSITIVITY_GPKG, layer=layer)['features']} features")

output/excluded_centres_2026.csv: 80 rows
output/excluded_centres_verification_2026.csv: 80 rows
output/excluded_centres_missingness_by_area.csv: 280 rows
output/unverified_centres_sensitivity_by_subzone.csv: 332 rows
output/childcare_centres_2026.gpkg layer planned: 13 features
output/childcare_centres_2026_sensitivity.gpkg layer plausible: 1806 features
output/childcare_centres_2026_sensitivity.gpkg layer upper_bound: 1815 features
